# Rival world-model benchmark on DROID-100 (Track B1)

This notebook trains the AdjointRWM reference model and four re-implemented rival world-model families on **exactly the same** DROID-100 data, split, inputs, parameter budget, optimiser budget, tuning budget and seeds, then scores them all with one method-blind function. It is milestone **B1** of `docs/plans/rival-benchmark-plan.md`; read that plan (§2 rivals, §3 fairness contract, §4 endpoints, §7 allowed statements) before running.

| Arm | What it is |
|---|---|
| `adjoint_rwm` | Reference: the pilot v1 architecture and stage-1 recipe, unchanged |
| `dreamerv3_rssm` | DreamerV3-style RSSM (block-GRU, 32 categorical latents, KL balancing, free nats) |
| `tdmpc2` | TD-MPC2-style decoder-free latent dynamics (SimNorm, ρ-weighted latent consistency) |
| `dino_wm` | DINO-WM-style block-causal predictor over frozen features (teacher-forced MSE) |
| `vjepa2_ac` | V-JEPA 2-AC-style predictor recipe (L1, normalised reps, 2-step rollout loss) |
| `persistence`, `ridge` | Classical baselines (no training / closed form) |

**Claim boundary.** Rivals are re-implementations at ~25 M prediction-path parameters, not the official systems at their published scale; name them "`<rival>`-style". This benchmark says nothing about H2 (adjoint vs direct critic); that is pilot v2's job.

**Rules that apply to every run** (`docs/production/colab_l4_operator_brief.md`): real DROID data only, no synthetic fallback; episode-level split before windowing; one immutable run directory; full checkpoint contract; failed seeds are reported, never replaced; the test split is evaluated once, after every arm is frozen.

**How to run.** Colab, GPU runtime (L4 is enough), Drive mounted. Run all cells. A long benchmark can span sessions: set `MAX_STEPS_PER_SESSION` in section 1, and when a cell stops with `PAUSED`, reconnect to the **same GPU type**, set `RESUME_RUN_ID` in section 1 and run all cells again. Finished jobs are skipped; paused jobs resume on the exact next batch.

In [ ]:
# ================================================================
# 0. Install, fetch the tested package, and pin its commit
# ================================================================
# tensorflow-metadata 1.21.0 (the latest) was generated with protobuf 6.31.1 but declares only protobuf>=4.25.2, so pip pairs it with the protobuf 5.x that
# TensorFlow 2.20 needs; tensorflow_datasets then fails inside its own import, silently, and exposes no load. Releases up to 1.17.3 do not check the runtime.
%pip -q install --upgrade "tensorflow-datasets>=4.9" rlds pyarrow "tensorflow-metadata<1.18"

import os
import subprocess
import sys

REPO_URL = 'https://github.com/romusking/para_001'
REPO_REF = 'main'  # branch, tag or commit SHA. Confirmatory runs pin a commit SHA.
REPO_DIR = '/content/para_001'

if not os.path.exists(REPO_DIR):
    # Private repository: authenticate first (for example a GitHub token from Colab secrets),
    # or upload a checkout of the repository to REPO_DIR.
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, REPO_DIR], check=True)
subprocess.run(['git', '-C', REPO_DIR, 'fetch', '--quiet', '--all'], check=False)


def resolve_ref(ref):
    for candidate in (f'origin/{ref}', ref):
        result = subprocess.run(
            ['git', '-C', REPO_DIR, 'rev-parse', '--verify', '--quiet', candidate + '^{commit}'],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            return result.stdout.strip()
    raise RuntimeError(f'Cannot resolve {ref!r} in {REPO_URL}.')


subprocess.run(['git', '-C', REPO_DIR, 'checkout', '--quiet', '--detach', resolve_ref(REPO_REF)], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
REPO_DIRTY = bool(subprocess.check_output(['git', '-C', REPO_DIR, 'status', '--porcelain'], text=True).strip())
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))

try:
    import adjointrwm
    import adjointrwm.benchmark
    import adjointrwm.models
    import adjointrwm.training  # noqa: F401
except ImportError as error:
    raise RuntimeError(
        f'{REPO_REF!r} ({REPO_COMMIT[:12]}) predates the benchmark modules. '
        'Set REPO_REF to the branch or commit that contains src/adjointrwm/models.'
    ) from error
print('adjointrwm', adjointrwm.__version__, 'at', REPO_COMMIT, '(uncommitted changes!)' if REPO_DIRTY else '')

In [ ]:
# ================================================================
# 1. Runtime, frozen configuration, immutable run directory
# ================================================================
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

import datetime
import json
import platform
import traceback
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from adjointrwm.io import append_jsonl, atomic_copy, atomic_write_json, atomic_write_text, file_record, sha256_json

assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
DEVICE = torch.device('cuda')
GPU_NAME = torch.cuda.get_device_name(0)
GPU_VRAM_GIB = torch.cuda.get_device_properties(0).total_memory / 1024**3
if not any(name in GPU_NAME for name in ('L4', 'A100', 'H100')):
    raise RuntimeError(f'Unsupported hardware profile: {GPU_NAME}')
torch.set_float32_matmul_precision('high')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


@dataclass(frozen=True)
class BenchmarkConfig:
    # Data: identical to the pilot (droid100_adjoint_20260929T070629Z).
    dataset_name: str = 'droid_100'
    dataset_data_dir: str = 'gs://gresearch/robotics'
    max_episodes: int = 100
    frame_stride: int = 2
    context_len: int = 8
    horizon: int = 4
    window_stride: int = 2
    pilot_run_id: str = 'droid100_adjoint_20260929T070629Z'  # split-parity reference

    # Frozen encoders. The shared evaluation target is always ResNet-18 (the pilot's).
    input_encoder: str = 'resnet18'  # or 'dinov2_vits14' (plan §8 / B4 sensitivity run)
    resnet_image_size: int = 128
    dinov2_image_size: int = 224
    dinov2_grid: int = 2
    embedding_batch_size: int = 64

    # Arms and the fairness contract (plan §3).
    arms: tuple = ('adjoint_rwm', 'dreamerv3_rssm', 'tdmpc2', 'dino_wm', 'vjepa2_ac')
    seeds: tuple = (0, 1, 2, 3, 4)
    seed_base: int = 20260928
    param_tolerance: float = 0.10
    steps: int = 1500
    batch_size: int = 64  # pilot's measured selection (systems_benchmark.json); same for every arm
    weight_decay: float = 0.05
    warmup_steps: int = 100
    eval_interval: int = 100
    checkpoint_interval: int = 250
    eval_batch_size: int = 128
    lr_grid: tuple = (1.0e-4, 3.0e-4, 1.0e-3)
    tuning_fraction: float = 0.30

    # Analysis (plan §4).
    margin: float = 0.02
    bootstrap_resamples: int = 5000
    shuffle_seed: int = 0
    eval_seed: int = 0


CFG = BenchmarkConfig()
CONFIG_DICT = asdict(CFG)
CONFIG_HASH = sha256_json(CONFIG_DICT)
TUNING_STEPS = int(CFG.tuning_fraction * CFG.steps)

RESUME_RUN_ID = None  # e.g. 'droid100_rivals_20261001T090000Z' to continue a paused run
MAX_STEPS_PER_SESSION = None  # optimiser steps this session may run (None = no limit); not part of the frozen config

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdjointRWM_Production')
SESSION_ID = datetime.datetime.now(datetime.UTC).strftime('%Y%m%dT%H%M%SZ')
if RESUME_RUN_ID:
    RUN_ID = RESUME_RUN_ID
    RUN_DIR = DRIVE_ROOT / 'runs' / RUN_ID
    stored = json.loads((RUN_DIR / 'config' / 'run_config.json').read_text())
    if stored['config_sha256'] != CONFIG_HASH:
        raise RuntimeError('This notebook config differs from the paused run. Resume with the identical config.')
    if stored['repo_commit'] != REPO_COMMIT:
        raise RuntimeError(f"The run used package commit {stored['repo_commit']}; set REPO_REF to it.")
    if stored['gpu'] != GPU_NAME:
        raise RuntimeError(f"The run started on {stored['gpu']}; reconnect to that GPU type (timings are never pooled).")
else:
    RUN_ID = 'droid100_rivals_' + SESSION_ID
    RUN_DIR = DRIVE_ROOT / 'runs' / RUN_ID
    if RUN_DIR.exists():
        raise RuntimeError(f'{RUN_DIR} already exists; run directories are immutable.')

LOCAL = Path('/content/adjoint_rwm_work') / RUN_ID
PATHS = {name: RUN_DIR / name for name in ('config', 'cache', 'jobs', 'artifacts', 'benchmarks', 'figures', 'logs', 'reports', 'tests')}
for path in [LOCAL, *PATHS.values()]:
    path.mkdir(parents=True, exist_ok=True)

SOURCE_MANIFEST = {
    'dataset': CFG.dataset_name,
    'data_dir': CFG.dataset_data_dir,
    'format': 'RLDS / TensorFlow Datasets',
    'licence': 'DROID: CC BY 4.0; attribute the DROID authors',
    'synthetic_fallback_allowed': False,
    'repo_url': REPO_URL,
    'repo_commit': REPO_COMMIT,
    'repo_dirty': REPO_DIRTY,
}
SOURCE_HASH = sha256_json(SOURCE_MANIFEST)
if not RESUME_RUN_ID:
    atomic_write_json(PATHS['config'] / 'run_config.json', {
        'run_id': RUN_ID, 'config': CONFIG_DICT, 'config_sha256': CONFIG_HASH, 'repo_commit': REPO_COMMIT,
        'gpu': GPU_NAME, 'notebook': 'notebooks/03-benchmarks/rival_world_models_droid100.ipynb',
    })
    atomic_write_json(PATHS['config'] / 'source_manifest.json', {**SOURCE_MANIFEST, 'source_sha256': SOURCE_HASH})

atomic_write_json(PATHS['config'] / f'environment_{SESSION_ID}.json', {
    'run_id': RUN_ID, 'session': SESSION_ID, 'python': sys.version, 'platform': platform.platform(),
    'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': GPU_NAME, 'gpu_vram_gib': GPU_VRAM_GIB,
    'precision': 'BF16 autocast, FP32 parameters and optimiser state',
})
atomic_write_text(PATHS['config'] / f'pip_freeze_{SESSION_ID}.txt',
                  subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))

METRICS_LOCAL = LOCAL / f'metrics_{SESSION_ID}.jsonl'


def log(record):
    append_jsonl(METRICS_LOCAL, {'time_utc': datetime.datetime.now(datetime.UTC).isoformat(), **record})


def sync_logs():
    if METRICS_LOCAL.exists():
        atomic_copy(METRICS_LOCAL, PATHS['logs'] / METRICS_LOCAL.name)


print('Run:', RUN_ID, '| resumed' if RESUME_RUN_ID else '| new', '| GPU:', GPU_NAME)
print('Run directory:', RUN_DIR)

In [ ]:
# ================================================================
# 2. Real DROID episodes -> frozen features (cached once per run), split parity
# ================================================================
import matplotlib.pyplot as plt

from adjointrwm.benchmark import split_hash
from adjointrwm.data import (
    IMAGE_KEYS, SPLITS, assignment_from_manifest, build_feature_cache, compare_splits, episode_split,
    feature_key, load_droid, persist_cache, restore_cache, split_sets, state_groups, write_cache_manifest,
)
from adjointrwm.features import DinoV2Embedder, ResNet18Embedder

CACHE_MANIFEST = PATHS['cache'] / 'cache_manifest.json'
LOCAL_CACHE = LOCAL / 'cache'

if CACHE_MANIFEST.exists():
    cache_manifest = json.loads(CACHE_MANIFEST.read_text())
    records = restore_cache(cache_manifest, LOCAL_CACHE)
    print(f'Restored {len(records)} cached episodes (hashes verified).')
else:
    import tensorflow_datasets as tfds

    raw_dataset, dataset_info = load_droid(CFG.dataset_name, CFG.dataset_data_dir)
    atomic_write_text(PATHS['config'] / 'tfds_builder_info.txt', str(dataset_info))
    embedders = {'resnet18': ResNet18Embedder(DEVICE, image_size=CFG.resnet_image_size, batch_size=CFG.embedding_batch_size)}
    if CFG.input_encoder == 'dinov2_vits14':
        embedders['dinov2_vits14'] = DinoV2Embedder(DEVICE, image_size=CFG.dinov2_image_size, grid=CFG.dinov2_grid,
                                                    batch_size=CFG.embedding_batch_size)
    elif CFG.input_encoder != 'resnet18':
        raise ValueError(f'unknown input encoder {CFG.input_encoder!r}')

    inspection = []

    def keep_for_inspection(episode_id, parsed):
        if len(inspection) < 8:
            inspection.append((episode_id, parsed['images']['exterior_image_1_left'][0],
                               parsed['images']['wrist_image_left'][0], parsed['states'], parsed['actions']))

    def droid_episodes():
        for index, episode in enumerate(raw_dataset.take(CFG.max_episodes)):
            yield index, episode, list(tfds.as_numpy(episode['steps']))

    records, skipped = build_feature_cache(
        droid_episodes(), embedders, LOCAL_CACHE, frame_stride=CFG.frame_stride,
        min_steps=CFG.context_len + CFG.horizon + 1, on_episode=keep_for_inspection,
    )
    if len(records) < 20:
        raise RuntimeError(f'Only {len(records)} valid episodes were cached; at least 20 are required.')
    records = persist_cache(records, PATHS['cache'] / 'episodes')
    cache_manifest = write_cache_manifest(records, skipped, CACHE_MANIFEST, {
        'encoders': {name: e.manifest() for name, e in embedders.items()}, 'source_sha256': SOURCE_HASH,
    })

    fig, axes = plt.subplots(len(inspection), 3, figsize=(14, 3 * len(inspection)))
    for row, (episode_id, exterior, wrist, states, actions) in enumerate(inspection):
        axes[row, 0].imshow(exterior); axes[row, 0].set_title(f'{episode_id} exterior'); axes[row, 0].axis('off')
        axes[row, 1].imshow(wrist); axes[row, 1].set_title('wrist'); axes[row, 1].axis('off')
        axes[row, 2].plot(states[:, 0], label='state[0]'); axes[row, 2].plot(actions[:, 0], label='action[0]')
        axes[row, 2].legend(); axes[row, 2].set_title('trajectory channels')
    plt.tight_layout()
    fig.savefig(PATHS['figures'] / 'episode_inspection.png', dpi=120)
    plt.show()
    plt.close(fig)
    del embedders
    torch.cuda.empty_cache()

# Episode-level split before any window exists; must equal the pilot's split.
assignment = episode_split([r['episode_id'] for r in records])
for record in records:
    record['split'] = assignment[record['episode_id']]
PILOT_MANIFEST = DRIVE_ROOT / 'runs' / CFG.pilot_run_id / 'config' / 'data_manifest.json'
if PILOT_MANIFEST.exists():
    SPLIT_PARITY = compare_splits(assignment, assignment_from_manifest(json.loads(PILOT_MANIFEST.read_text())))
    if not SPLIT_PARITY['matches']:
        raise RuntimeError(f'The episode split differs from the pilot run: {SPLIT_PARITY}')
else:
    SPLIT_PARITY = {'matches': None, 'reason': f'{PILOT_MANIFEST} not found; same rule, not compared'}
sets = split_sets(assignment)
assert sets['train'].isdisjoint(sets['validation']) and sets['train'].isdisjoint(sets['test'])
assert sets['validation'].isdisjoint(sets['test'])

STATE_KEYS_USED, STATE_DIMS = records[0]['state_keys'], records[0]['state_dims']
GROUPS = state_groups(STATE_KEYS_USED, STATE_DIMS)
STABLE_FIELDS = ('episode_index', 'episode_id', 'cached_sha256', 'length', 'split', 'instruction')
DATA_MANIFEST = {
    'source_sha256': SOURCE_HASH,
    'frame_stride': CFG.frame_stride,
    'state_keys': STATE_KEYS_USED,
    'state_dims': STATE_DIMS,
    'action_dim': records[0]['action_dim'],
    'encoders': cache_manifest['encoders'],
    'episodes': [{k: r[k] for k in STABLE_FIELDS} for r in sorted(records, key=lambda r: r['episode_id'])],
    'skipped': cache_manifest['skipped'],
    'split_counts': {s: len(v) for s, v in sets.items()},
}
DATA_MANIFEST_HASH = sha256_json(DATA_MANIFEST)
SPLIT_HASH = split_hash(assignment)
atomic_write_json(PATHS['config'] / 'data_manifest.json', {**DATA_MANIFEST, 'data_manifest_sha256': DATA_MANIFEST_HASH})
DATA_AUDIT = {
    'valid_episode_count': len(records), 'split_counts': DATA_MANIFEST['split_counts'], 'split_parity_with_pilot': SPLIT_PARITY,
    'real_image_payloads_verified': True, 'split_disjointness_verified': True, 'synthetic_fallback_used': False, 'status': 'PASS',
}
atomic_write_json(PATHS['tests'] / 'data_audit.json', DATA_AUDIT)
print(json.dumps(DATA_AUDIT, indent=2))

In [ ]:
# ================================================================
# 3. Windows, normalisation and classical baselines
# ================================================================
from adjointrwm.benchmark import windows_hash
from adjointrwm.data import WindowDataset, WindowSpec, fit_normaliser
from adjointrwm.eval import RidgeForecaster, derangement, persistence_forecast, prediction_frame, stack_windows

SPEC = WindowSpec(CFG.context_len, CFG.horizon, CFG.window_stride)
BY_SPLIT = {s: [r for r in records if r['split'] == s] for s in SPLITS}


def load_arrays(record, keys=('states', 'actions')):
    with np.load(record['cached_path']) as episode:
        return {k: episode[k] for k in keys}


train_arrays = [load_arrays(r) for r in BY_SPLIT['train']]
NORMALISATION = fit_normaliser([a['states'] for a in train_arrays], [a['actions'] for a in train_arrays])
np.savez(PATHS['config'] / 'normalisation.npz', **NORMALISATION)

INPUT_KEYS = tuple(feature_key(CFG.input_encoder, k) for k in IMAGE_KEYS)
TARGET_KEYS = tuple(feature_key('resnet18', k) for k in IMAGE_KEYS)
DATASETS = {
    s: WindowDataset(BY_SPLIT[s], SPEC, NORMALISATION, input_visual_keys=INPUT_KEYS,
                     target_visual_keys=TARGET_KEYS, visual_layout='tokens')
    for s in SPLITS
}
TEST_WINDOWS_HASH = windows_hash(DATASETS['test'].episode_ids(), [start for _, start in DATASETS['test'].index])
print('Windows:', {s: len(d) for s, d in DATASETS.items()})

STACK_KEYS = ['context_state', 'context_action', 'future_actions', 'target_state']
STACKED = {s: stack_windows(DATASETS[s], keys=STACK_KEYS) for s in SPLITS}
TEST = STACKED['test']
SHUFFLE = derangement(len(DATASETS['test']), CFG.shuffle_seed)

ridge = RidgeForecaster().fit(STACKED['train'], STACKED['validation'])
BASELINE_PREDICTIONS = {
    ('persistence', 'nominal'): persistence_forecast(TEST['context_state'], CFG.horizon),
    ('ridge', 'nominal'): ridge.predict(TEST['context_state'], TEST['context_action'], TEST['future_actions']),
    ('ridge', 'actions_shuffled'): ridge.predict(TEST['context_state'], TEST['context_action'], TEST['future_actions'][SHUFFLE]),
}
atomic_write_json(PATHS['artifacts'] / 'ridge_baseline.json', {
    'lambda': ridge.lambda_, 'validation_mse_by_lambda': ridge.validation_mse_, 'history': ridge.history,
})

# Deterministic baselines are repeated under every seed label so paired comparisons line up.
FRAMES = []
for (arm, condition), prediction in BASELINE_PREDICTIONS.items():
    for seed in CFG.seeds:
        FRAMES.append(prediction_frame(
            arm=arm, seed=seed, episode_ids=TEST['episode_id'], window_starts=TEST['window_start'],
            pred_state=prediction, target_state=TEST['target_state'], state_std=NORMALISATION['state_std'],
            groups=GROUPS, condition=condition,
        ))
print('Ridge lambda (validation):', ridge.lambda_)

In [ ]:
# ================================================================
# 4. Arms at a matched prediction-path parameter budget
# ================================================================
from adjointrwm.models import RECIPE_DEFAULTS, REFERENCE_ARM, ArmDims, build_arm, count_prediction_parameters, match_width

with np.load(BY_SPLIT['train'][0]['cached_path']) as episode:
    token_shape = episode[INPUT_KEYS[0]].shape[1:]  # (D,) or (P, D) per camera
    target_dim = sum(episode[k].reshape(len(episode[k]), -1).shape[1] for k in TARGET_KEYS)
tokens_per_camera = 1 if len(token_shape) == 1 else token_shape[0]
DIMS = ArmDims(
    state_dim=records[0]['state_dim'], action_dim=records[0]['action_dim'],
    visual_tokens=len(INPUT_KEYS) * tokens_per_camera, visual_token_dim=token_shape[-1],
    target_visual_dim=target_dim, context_len=CFG.context_len, horizon=CFG.horizon,
)
assert REFERENCE_ARM in CFG.arms
REFERENCE_PARAMETERS = count_prediction_parameters(REFERENCE_ARM, DIMS)
ARM_SPECS = {}
for arm in CFG.arms:
    if arm == REFERENCE_ARM:
        spec = {'width': None, 'parameters': REFERENCE_PARAMETERS, 'relative_gap': 0.0, 'within_tolerance': True}
    else:
        spec = match_width(arm, DIMS, REFERENCE_PARAMETERS, CFG.param_tolerance)
    if not spec['within_tolerance']:
        raise RuntimeError(f'{arm} cannot be matched to the reference parameter budget: {spec}')
    ARM_SPECS[arm] = {**spec, 'recipe': RECIPE_DEFAULTS[arm]}
atomic_write_json(PATHS['config'] / 'arm_manifest.json', {'dims': DIMS.to_dict(), 'reference': REFERENCE_ARM,
                                                          'reference_parameters': REFERENCE_PARAMETERS, 'arms': ARM_SPECS})
print(pd.DataFrame(ARM_SPECS).T[['width', 'parameters', 'relative_gap']].to_string())

In [ ]:
# ================================================================
# 5. Resumable jobs: equal validation-only LR search, then 5 paired seeds per arm
# ================================================================
from adjointrwm.training import TrainConfig, seed_everything, train_job

SESSION = {'steps_left': MAX_STEPS_PER_SESSION}


def job_identity(arm, seed, tag):
    return {'run_id': RUN_ID, 'arm': f'{tag}:{arm}', 'seed': int(seed), 'config_hash': CONFIG_HASH,
            'data_manifest_hash': DATA_MANIFEST_HASH, 'source_hash': SOURCE_HASH}


def build_model(arm, seed):
    seed_everything(CFG.seed_base + seed)  # initialisation depends on the seed only
    return build_arm(arm, DIMS, width=ARM_SPECS[arm]['width'])


def train_config(arm, lr, steps, seed):
    return TrainConfig(
        steps=steps, batch_size=CFG.batch_size, lr=lr, weight_decay=CFG.weight_decay,
        warmup_steps=min(CFG.warmup_steps, max(1, steps // 5)), grad_clip=RECIPE_DEFAULTS[arm]['grad_clip'],
        eval_interval=CFG.eval_interval, checkpoint_interval=CFG.checkpoint_interval, seed=CFG.seed_base + seed,
        amp=True, eval_batch_size=CFG.eval_batch_size,
    )


def job_dir(tag, arm, seed):
    return PATHS['jobs'] / tag / arm / f'seed_{seed}'


def run_job(arm, seed, tag, lr, steps):
    persist = job_dir(tag, arm, seed)
    if (persist / 'FAILED.json').exists():
        return json.loads((persist / 'FAILED.json').read_text())
    if (persist / 'DONE.json').exists():
        return json.loads((persist / 'DONE.json').read_text())
    if SESSION['steps_left'] is not None and SESSION['steps_left'] <= 0:
        return {'status': 'PAUSED', 'arm': arm, 'seed': seed}
    model = build_model(arm, seed)
    try:
        summary = train_job(
            model, DATASETS['train'], DATASETS['validation'], train_config(arm, lr, steps, seed),
            identity=job_identity(arm, seed, tag), local_dir=LOCAL / 'jobs' / tag / arm / f'seed_{seed}',
            persist_dir=persist, device=DEVICE, max_steps_this_session=SESSION['steps_left'],
            log=lambda record: log({'job': f'{tag}/{arm}/seed_{seed}', **record}),
            keep_best=tag == 'main',  # LR-search weights are never reused; their hashes stay in DONE.json
        )
    except FloatingPointError as error:
        summary = {**job_identity(arm, seed, tag), 'status': 'FAILED', 'error': repr(error), 'traceback': traceback.format_exc()}
        atomic_write_json(persist / 'FAILED.json', summary)
    if SESSION['steps_left'] is not None:
        SESSION['steps_left'] -= summary.get('session_steps', 0)
    del model
    torch.cuda.empty_cache()
    sync_logs()
    print(f"{tag:>18} {arm:>15} seed {seed}: {summary['status']}", summary.get('best', {}).get('validation_score', ''))
    return summary


def require_terminal(summaries, stage):
    pending = [(a, s) for (a, s), v in summaries.items() if v['status'] not in ('DONE', 'FAILED')]
    if pending:
        raise RuntimeError(f'{stage}: {len(pending)} job(s) PAUSED. Set RESUME_RUN_ID = {RUN_ID!r} and run all cells again.')


# 5a. Equal learning-rate search (validation only, seed index 0, 30 % of the steps).
TUNING = {}
for arm in CFG.arms:
    for lr in CFG.lr_grid:
        TUNING[(arm, lr)] = run_job(arm, CFG.seeds[0], f'tuning/lr_{lr:g}', lr, TUNING_STEPS)
require_terminal(TUNING, 'LR search')
BEST_LR = {}
for arm in CFG.arms:
    trials = {lr: TUNING[(arm, lr)] for lr in CFG.lr_grid}
    finished = {lr: t['best']['validation_score'] for lr, t in trials.items() if t['status'] == 'DONE'}
    if not finished:
        raise RuntimeError(f'Every LR trial of {arm} failed; the arm cannot be compared (report it).')
    BEST_LR[arm] = min(finished, key=finished.get)
atomic_write_json(PATHS['artifacts'] / 'tuning.json', {
    'lr_grid': list(CFG.lr_grid), 'tuning_steps': TUNING_STEPS, 'selected_lr': BEST_LR,
    'trials': [{'arm': a, 'lr': lr, 'status': t['status'], 'validation_rmse': t.get('best', {}).get('validation_score')}
               for (a, lr), t in TUNING.items()],
})
print('Selected learning rates:', BEST_LR)

# 5b. Main runs: every arm x every paired seed, with its validation-selected LR.
MAIN = {(arm, seed): run_job(arm, seed, 'main', BEST_LR[arm], CFG.steps) for arm in CFG.arms for seed in CFG.seeds}
require_terminal(MAIN, 'Main runs')
print(pd.DataFrame([{'arm': a, 'seed': s, 'status': v['status'], 'best_step': v.get('best', {}).get('step'),
                     'validation_rmse': v.get('best', {}).get('validation_score')} for (a, s), v in MAIN.items()]).to_string())

In [ ]:
# ================================================================
# 6. Test evaluation (once, after every arm is frozen) and systems cost
# ================================================================
from adjointrwm.eval import effective_rank
from adjointrwm.training import PermutedActions, load_best_weights, measure_latency_ms, move_batch, predict_dataset

SHUFFLED_TEST = PermutedActions(DATASETS['test'], SHUFFLE)
EVALUATED, SYSTEMS, RANKS = {}, {}, []
latency_batch = move_batch(torch.utils.data.default_collate([DATASETS['test'][i] for i in range(CFG.batch_size)]), DEVICE)
for (arm, seed), summary in MAIN.items():
    if summary['status'] != 'DONE':
        continue
    model = build_model(arm, seed).to(DEVICE)
    load_best_weights(model, job_dir('main', arm, seed), identity=job_identity(arm, seed, 'main'))
    model.eval()
    for condition, dataset in (('nominal', DATASETS['test']), ('actions_shuffled', SHUFFLED_TEST)):
        pred = predict_dataset(model, dataset, CFG.eval_batch_size, DEVICE, eval_seed=CFG.eval_seed, with_latents=condition == 'nominal')
        FRAMES.append(prediction_frame(
            arm=arm, seed=seed, episode_ids=pred['episode_id'], window_starts=pred['window_start'],
            pred_state=pred['state_mean'], target_state=pred['target_state'], state_std=NORMALISATION['state_std'],
            groups=GROUPS, pred_visual=pred['visual'], target_visual=pred['target_visual'], pred_logvar=pred['state_logvar'],
            condition=condition,
        ))
        if condition == 'nominal':
            EVALUATED[(arm, seed)] = windows_hash(pred['episode_id'], pred['window_start'])
            RANKS.append({'arm': arm, 'seed': seed, 'latent_effective_rank': effective_rank(pred['latent']),
                          'latent_dim': int(pred['latent'].shape[1])})
    if seed == CFG.seeds[0]:
        with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            SYSTEMS[arm] = {
                'predict_latency_batch': CFG.batch_size,
                'predict_latency': measure_latency_ms(lambda: model.predict(latency_batch), DEVICE),
            }
    del model
    torch.cuda.empty_cache()

for arm in CFG.arms:
    done = [MAIN[(arm, s)] for s in CFG.seeds if MAIN[(arm, s)]['status'] == 'DONE']
    rates = [d['session_steps_per_second'] for d in done if d.get('session_steps_per_second')]
    peaks = [d['peak_vram_gib'] for d in done if d.get('peak_vram_gib') is not None]
    SYSTEMS.setdefault(arm, {}).update({
        'prediction_parameters': ARM_SPECS[arm]['parameters'],
        'train_steps_per_second_median': float(np.median(rates)) if rates else None,
        'train_steps_per_second_note': 'per-session rate of the session that finished each job',
        'peak_training_vram_gib_max': float(max(peaks)) if peaks else None,
        'gpu': GPU_NAME,
    })
atomic_write_json(PATHS['benchmarks'] / 'systems_benchmark.json', SYSTEMS)

ERRORS = pd.concat(FRAMES, ignore_index=True)
for column in ('arm', 'split', 'condition', 'episode_id'):
    ERRORS[column] = ERRORS[column].astype('category')
float_columns = ERRORS.select_dtypes('float64').columns
ERRORS[float_columns] = ERRORS[float_columns].astype('float32')
ERRORS.to_parquet(PATHS['artifacts'] / 'prediction_errors.parquet', index=False)
pd.DataFrame(RANKS).to_csv(PATHS['artifacts'] / 'latent_effective_rank.csv', index=False)
print(pd.DataFrame(SYSTEMS).T.to_string())

In [ ]:
# ================================================================
# 7. Paired comparisons, fairness contract, report
# ================================================================
from adjointrwm.benchmark import check_fairness
from adjointrwm.eval import classify_relative_difference, paired_relative_difference, summarize_frame

ERRORS = pd.read_parquet(PATHS['artifacts'] / 'prediction_errors.parquet')
for column in ('arm', 'condition', 'episode_id', 'split'):
    ERRORS[column] = ERRORS[column].astype(str)
SUMMARY = summarize_frame(ERRORS)
SUMMARY.to_csv(PATHS['artifacts'] / 'summary_by_arm_seed.csv', index=False)
NOMINAL = ERRORS[ERRORS['condition'] == 'nominal']

COMPARISONS = {}
rivals = [a for a in CFG.arms if a != REFERENCE_ARM and any(MAIN[(a, s)]['status'] == 'DONE' for s in CFG.seeds)]
metrics = ['mse_norm'] + [f'mse_native_{g}' for g in GROUPS]
for rival in rivals + ['persistence', 'ridge']:
    for metric in metrics:
        result = paired_relative_difference(NOMINAL, REFERENCE_ARM, rival, metric=metric,
                                            num_resamples=CFG.bootstrap_resamples, seed=0)
        result['classification'] = classify_relative_difference(result['ci_low'], result['ci_high'], CFG.margin)
        result['primary'] = metric == 'mse_norm'
        COMPARISONS[f'{REFERENCE_ARM}_vs_{rival}:{metric}'] = result

# Action sensitivity: how much worse each arm gets when future actions belong to another window.
by = SUMMARY.set_index(['arm', 'seed', 'condition'])['rmse_norm']
ACTION_SENSITIVITY = {}
for arm in [REFERENCE_ARM, *rivals, 'ridge']:
    ratios = [by[(arm, s, 'actions_shuffled')] / by[(arm, s, 'nominal')] - 1.0
              for s in CFG.seeds if (arm, s, 'actions_shuffled') in by.index and (arm, s, 'nominal') in by.index]
    ACTION_SENSITIVITY[arm] = {'relative_rmse_increase_per_seed': ratios,
                               'mean': float(np.mean(ratios)) if ratios else None}

FAIRNESS_RECORDS = {}
for arm in CFG.arms:
    finished = [s for s in CFG.seeds if MAIN[(arm, s)]['status'] == 'DONE']
    FAIRNESS_RECORDS[arm] = {
        'data_manifest_hash': DATA_MANIFEST_HASH, 'split_hash': SPLIT_HASH, 'window_spec': SPEC.to_dict(),
        'test_windows_hash': (EVALUATED[(arm, finished[0])] if finished and all(EVALUATED[(arm, s)] == TEST_WINDOWS_HASH for s in finished)
                              else 'MISMATCH_OR_MISSING'),
        'input_encoder': CFG.input_encoder, 'target_encoder': 'resnet18', 'steps': CFG.steps, 'batch_size': CFG.batch_size,
        'seeds': list(CFG.seeds), 'lr_grid': list(CFG.lr_grid), 'tuning_steps': TUNING_STEPS, 'gpu': GPU_NAME,
        'prediction_parameters': ARM_SPECS[arm]['parameters'],
        'seeds_finished': finished, 'seeds_failed': [s for s in CFG.seeds if MAIN[(arm, s)]['status'] == 'FAILED'],
    }
FAIRNESS = check_fairness(FAIRNESS_RECORDS, REFERENCE_ARM, CFG.param_tolerance)
atomic_write_json(PATHS['tests'] / 'fairness_contract.json', {'records': FAIRNESS_RECORDS, **FAIRNESS})

# Raw table first, then the figure drawn from it.
by_h = (NOMINAL.groupby(['arm', 'seed', 'h'])['mse_norm'].mean() ** 0.5).rename('rmse_norm').reset_index()
by_h.to_csv(PATHS['figures'] / 'rmse_by_horizon.csv', index=False)
fig, ax = plt.subplots(figsize=(8, 5))
for arm, group in by_h.groupby('arm'):
    mean = group.groupby('h')['rmse_norm'].mean()
    ax.plot(mean.index, mean.values, marker='o', label=arm)
    ax.scatter(group['h'], group['rmse_norm'], s=8, alpha=0.4)
ax.set_xlabel('horizon step'); ax.set_ylabel('test RMSE (normalised state)'); ax.set_xticks(range(1, CFG.horizon + 1))
ax.set_title(f'{RUN_ID}: mean over seeds (dots = seeds)'); ax.legend()
fig.savefig(PATHS['figures'] / 'rmse_by_horizon.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close(fig)

primary = {k: v for k, v in COMPARISONS.items() if v['primary']}
REPORT = {
    'run_id': RUN_ID,
    'status': {
        'data': DATA_AUDIT['status'],
        'fairness_contract': 'PASS' if FAIRNESS['passed'] else 'FAIL',
        'jobs': {f'{a}/seed_{s}': v['status'] for (a, s), v in MAIN.items()},
    },
    'config_hash': CONFIG_HASH, 'data_manifest_hash': DATA_MANIFEST_HASH, 'source_hash': SOURCE_HASH,
    'split_parity_with_pilot': SPLIT_PARITY,
    'selected_lr': BEST_LR,
    'primary_endpoint': 'test RMSE of normalised proprio state, mean over horizon steps (plan §4)',
    'comparisons': COMPARISONS,
    'classification_margin': CFG.margin,
    'action_sensitivity': ACTION_SENSITIVITY,
    'systems': SYSTEMS,
    'claim_boundary': ('Rivals are re-implementations ("-style") at a matched ~25M prediction-path budget on DROID-100 '
                       f"({DATA_MANIFEST['split_counts']['test']} test episodes). No H2, task-success or SOTA claim follows."),
}
atomic_write_json(PATHS['reports'] / 'acceptance_report.json', REPORT)

lines = [f'# Rival world-model benchmark `{RUN_ID}`', '', f"Fairness contract: **{REPORT['status']['fairness_contract']}**", '',
         '| Comparison (reference vs rival) | Reference RMSE | Rival RMSE | Relative diff. | 95 % CI | Classification |',
         '|---|---:|---:|---:|---|---|']
for name, c in primary.items():
    lines.append(f"| {name.split(':')[0]} | {c['reference_rmse']:.4f} | {c['rival_rmse']:.4f} | {c['relative_difference']:+.3f} | "
                 f"[{c['ci_low']:+.3f}, {c['ci_high']:+.3f}] | {c['classification']} |")
lines += ['', 'Negative relative difference = the reference (AdjointRWM) has lower RMSE.', '', REPORT['claim_boundary']]
atomic_write_text(PATHS['reports'] / 'run_summary.md', '\n'.join(lines) + '\n')
sync_logs()

if FAIRNESS['passed'] and all(v['status'] in ('DONE', 'FAILED') for v in MAIN.values()):
    atomic_write_text(RUN_DIR / 'COMPLETE', datetime.datetime.now(datetime.UTC).isoformat() + '\n')
else:
    print('Not marking COMPLETE: fairness contract failed or jobs are unfinished. See tests/fairness_contract.json.')
print('\n'.join(lines))
print('\nRun directory:', RUN_DIR)

## Reading and importing the result

- **Every classification is reported**, including `rival_better` and `inconclusive`. The allowed wording for each outcome is in `docs/plans/rival-benchmark-plan.md` §7.
- **A failed fairness contract is a result too.** Don't re-run with a changed contract to make it pass; write down why it failed.
- **Action sensitivity near zero** means an arm ignores the actions. It can still have a low RMSE by extrapolating the context, so read it alongside the primary endpoint.
- **Import** the small artefacts with the `import-run` skill: `config/*.json`, `tests/*.json`, `artifacts/*.json|csv|parquet`, `benchmarks/*.json`, `figures/*`, `reports/*`. Check `prediction_errors.parquet` is under 5 MB. Checkpoints and the feature cache stay on Drive; list their SHA-256 values in the run README.
- **Then** write the research note with the `research-note` skill and run `claim-check` on it.